In [ ]:
import os
import json
from dotenv import load_dotenv
from crewai import Agent, Task, Crew, Process, LLM
from crewai_tools import SerperDevTool, ScrapeWebsiteTool
from pathlib import Path
from pydantic import BaseModel, Field
from crewai.skills import discover_skills, activate_skill

# Patch cache breakpoint for providers like Groq/Ollama if needed
import crewai.llms.cache as _crewai_cache
_crewai_cache.mark_cache_breakpoint = lambda msg: msg

# Monkey patch LLM.supports_function_calling to return False for Groq models
original_supports_function_calling = LLM.supports_function_calling

def patched_supports_function_calling(self) -> bool:
    model_name = self.model or ""
    provider = getattr(self, "provider", None) or self._get_custom_llm_provider()
    if "groq" in model_name.lower() or provider == "groq":
        return False
    return original_supports_function_calling(self)

LLM.supports_function_calling = patched_supports_function_calling

# Load environment variables
load_dotenv()
SERPER_API_KEY = os.getenv("SERPER_API_KEY")
os.environ["SERPER_API_KEY"] = SERPER_API_KEY or ""

# Initialize tools required for Phase 1 Desirability market analysis
search_tool = SerperDevTool(api_key=SERPER_API_KEY)
scrape_tool = ScrapeWebsiteTool()

# Configure LLM (Local Ollama Qwen 2.5 7B)
llm = LLM(
    model="bonsai-8b", 
    base_url="http://127.0.0.1:1234/v1", 
    api_key="lm-studio",
    temperature=0.1,
)

# Discover and activate skills
skills = discover_skills(Path("./skills"))
activated = [activate_skill(s) for s in skills]

# Define the Pydantic models for JSON output
class RefinedIdea(BaseModel):
    customer_segment: str = Field(description="A refined description of the target customer segment, e.g., 'Undergraduate students at PES University with low CGPA risk.'")
    qualified_problem: str = Field(description="The qualified problem being solved, e.g., 'Missing internal assessment (ISA) deadlines because multi-channel notifications cause administrative blindness.'")
    consequence: str = Field(description="The negative consequence of not solving the problem, e.g., 'Direct loss of 5-10% of internal marks, delaying graduation.'")
    proposed_solution: str = Field(description="The proposed technical solution, e.g., 'An automated WhatsApp-based scraper that extracts personalized deadlines from the PES LMS and sends a daily morning Action Agenda.'")

class TipsValidatedMetrics(BaseModel):
    timely_factor: str = Field(description="The timely factor/urgency metric, explaining why this is an active daily issue right now.")
    importance_metric: str = Field(description="The importance metric, explaining the severe impact of the consequence on the user.")
    profitability_pivot: str = Field(description="The profitability pivot, explaining the business model and who pays.")
    solvability_constraint: str = Field(description="The solvability constraint/implementation feasibility, explaining how the team can build it with simple/accessible tools.")

class DFAOutput(BaseModel):
    refined_idea: RefinedIdea
    tips_validated_metrics: TipsValidatedMetrics

# Define the Phase 1 Desirability Evaluation Agent
desirability_agent = Agent(
    role="Desirability Evaluation Agent",
    goal="Determine whether the proposed solution addresses a genuine user need and whether sufficient market demand exists.",
    backstory=(
        """You are an expert market research analyst and user experience strategist. You MUST use the Search tool and ScrapeWebsite tool for EVERY task.
    Do NOT answer from memory or prior knowledge.
    Your first action must always be a tool call.
    If you have not searched the web, your answer is incomplete."""
    ),
    llm=llm,
    tools=[search_tool, scrape_tool],
    verbose=True,
    skills=[activated[0]],
    max_iter=4
)

# Define the Desirability Task mapped exactly to system documentation outputs
desirability_task = Task(
    description="{desirability}",
    expected_output=(
        """A formal text-formatted 'Desirability Analysis Report' containing:
        1. User Demand Analysis (validating target user pain points and problem severity).
        2. Market Demand Assessment (current industry search interest and growth indicators).
        3. Competitor Analysis (gaps, weaknesses, or friction in existing products/alternatives).
        4. Opportunity Identification (clear statement on why this solution is or is not desired by the market).
        keep the output under 600 words"""
    ),
    agent=desirability_agent,
    async_execution=True
)

# Define Feasibility Agent and Task
feasibility_agent = Agent(
    role="Feasibility Evaluation Agent",
    goal="Evaluate the feasibility of a startup idea strictly from the Feasibility dimension of the DFV framework.",
    backstory=(
        """You are an expert technical architect, systems analyst, and execution strategist. You MUST use the Search tool and ScrapeWebsite tool for EVERY task.
    Do NOT answer from memory or prior knowledge.
    Your first action must always be a tool call.
    If you have not searched the web, your answer is incomplete."""
    ),
    llm=llm,
    tools=[search_tool, scrape_tool],
    verbose=True,
    skills=[activated[2]],
    max_iter=4
)

feasibility_task = Task(
    description="{feasibility}",
    expected_output=(
        "A plain-language Feasibility Evaluation containing:\n"
        "1. A short feasibility opinion.\n"
        "2. Main technical and operational challenges.\n"
        "3. Required tools, stack, or infrastructure.\n"
        "4. Suggestions to improve or simplify the idea if needed.\n"
        "5. Practical next steps for implementation.\n"
        "Do not include any score, rating, grade, or percentage. keep the output under 600 words"
    ),
    agent=feasibility_agent,
    async_execution=True
)

# Define Viability Agent and Task
viability_agent = Agent(
    role="Viability Evaluation Agent",
    goal="Determine whether the proposed solution can generate sustainable business value and long-term growth.",
    backstory=(
        """You are an expert startup strategist, business consultant, and commercialization analyst. You MUST use the Search tool and ScrapeWebsite tool for EVERY task.
    Do NOT answer from memory or prior knowledge.
    Your first action must always be a tool call.
    If you have not searched the web, your answer is incomplete."""
    ),
    llm=llm,
    tools=[search_tool, scrape_tool],
    verbose=True,
    skills=[activated[3]],
    max_iter=4
)

viability_task = Task(
    description="{viability}",
    expected_output=(
        "A Viability Analysis Report containing:\n"
        "1. Business Model Analysis\n"
        "2. Revenue Opportunities\n"
        "3. Customer Segment Analysis\n"
        "4. Cost Considerations\n"
        "5. Sustainability Assessment\n"
        "6. Final Viability Conclusion\n"
        "keep the output under 600 words"
    ),
    agent=viability_agent,
    async_execution=True
)

# Define Decision and Risk Assessment Agent and Task
dfv_risk_decision_agent = Agent(
    role="Internal DFV Decision and Risk Assessment Engine",
    goal="Identify hidden risks across project dimensions and aggregate all findings into a final project readiness decision.",
    backstory="You are an expert venture risk analyst and product strategist.",
    verbose=True,
    skills=[activated[1]],
    llm=llm
)

dfv_decision_task = Task(
    description=(
        """Review the reports provided in your context thoroughly from the Desirability,
        Feasibility, and Viability evaluation phases. Synthesize these findings to construct
        a structured assessment of the project idea, filling in the required JSON fields.

        Specifically:
        1. refined_idea:
           - customer_segment: The precise group of users experiencing the problem.
           - qualified_problem: The specific pain point or problem being addressed.
           - consequence: The direct negative consequence of the problem if left unsolved.
           - proposed_solution: The proposed product/solution.
        2. tips_validated_metrics:
           - timely_factor: Why this is a timely problem to solve now (e.g. changes in evaluation weightage or policies).
           - importance_metric: Why this problem is highly important/consequential (e.g. impact on placements or graduation).
           - profitability_pivot: The business/revenue model pivot or approach (e.g. B2B2C parent-pay model vs student-pay).
           - solvability_constraint: The technical feasibility constraint showing it is solvable with simple tools."""
    ),
    expected_output="A structured JSON object matching the DFAOutput schema with refined_idea and tips_validated_metrics.",
    context=[desirability_task, feasibility_task, viability_task],
    agent=dfv_risk_decision_agent,
    output_json=DFAOutput
)

# Define input ideas
pes_lms = {
    "desirability": (
        "Undergraduate students at PES University face a major issue with missing internal assessment (ISA) deadlines. "
        "Because notifications are sent across multiple channels like WhatsApp, LMS, and Email, it causes administrative blindness for students. "
        "Consequently, students suffer a direct loss of 5-10% of their internal marks, which delays graduation or severely impacts final year placement eligibility. "
        "Tracking deadlines has become a daily active issue because PES recently increased the weightage of continuous evaluation."
    ),
    "feasibility": (
        "The proposed solution is an automated WhatsApp-based scraper that extracts personalized deadlines from the PES LMS and sends a daily morning 'Action Agenda'. "
        "The implementation is constrained such that the student team can build a basic Python scraper and use a local WhatsApp API wrapper without needing advanced infrastructure."
    ),
    "viability": (
        "The project was initially planned as a student subscription model, but since students are price-sensitive, "
        "it switched to a B2B2C model where anxious parents pay a small monthly fee of Rs. 100 to receive academic risk alerts about their ward's upcoming deadlines."
    )
}

blnkt={
    "desirability":""" Analyze the following student project proposal:
        - Customer Problem: Urban consumers need immediate access to groceries and essentials without spending time traveling to stores
        - Target Audience: Millennials, Gen Z, busy professionals, and students in metro cities aged 18-40
        - Key Value Proposition: 10-minute delivery guarantee, real-time order tracking, wide product selection
        - User Pain Points Solved: Time savings, convenience for impulse purchases, avoids crowded stores
        - Market Demand Indicators: High adoption rate in urban India, 4.2+ app rating, repeat usage frequency
        - Emotional Drivers: Convenience, instant gratification, time flexibility
                                          """,
    "feasibility":""" The following is a startup/project idea submitted by a user:
        - Technology Stack: React Native mobile apps, cloud infrastructure, inventory management systems, route optimization algorithms
        - Infrastructure Model: Dark stores (micro-warehouses) located 2-3km from target customers, 500+ sq ft each
        - Logistics: Proprietary delivery fleet of 5,000+ delivery partners with GPS tracking
        - Supply Chain: Partnerships with 10,000+ local retailers and wholesale distributors
        - Operational Capabilities: Real-time demand forecasting, automated inventory replenishment, dynamic routing
        - Technical Challenges: Inventory accuracy, delivery time optimization, peak-hour scalability, cold chain for perishables
        - Resource Requirements: Capital investment for dark store network, technology development team, delivery workforce""",
    "viability":"""
        Analyze the business viability of the following project proposal:
        - Revenue Model:
          * Delivery fees (₹29-₹59 per order)
          * Platform commissions from sellers (15-25%)
          * Advertising fees from brands
          * Blinkit Prime membership (₹99/month)

        - Cost Structure:
          * Dark store operational costs (rent, staffing, inventory)
          * Delivery partner payments (₹40-₹60 per delivery)
          * Technology and infrastructure costs
          * Marketing and customer acquisition costs

        - Market Size: India quick commerce market = $3B in 2024, projected $5-7B by 2025
        - Unit Economics: Average order value ₹300-₹500, order frequency 2-3 times/week per customer
        - Competitive Position: Market leader in 10-minute delivery, competes with Swiggy Instamart, Zepto, BigBasket
        - Profitability Status: Contribution margin positive as of 2024 (reported by Zomato)
        - Growth Trajectory: 300+ cities, 50M+ active users, 20% monthly growth
        """
}

# Assemble the Crew
crew = Crew(
    agents=[desirability_agent, feasibility_agent, viability_agent, dfv_risk_decision_agent],
    tasks=[desirability_task, feasibility_task, viability_task, dfv_decision_task],
    process=Process.sequential,
    verbose=True
)

print("Running crew execution on PES LMS idea...")
result = await crew.kickoff_async(inputs=pes_lms)
print("\n--- FINAL DFA JSON OUTPUT ---\n")
try:
    print(json.dumps(json.loads(result.raw), indent=2))
except Exception:
    print(result.raw)
